# SAM Token Merging Observer

This notebook patches the SAM image encoder, captures block-level token features, and compares spatial edge intensity against cosine-similarity structure.

In [1]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib.colors import ListedColormap
from segment_anything import SamPredictor, sam_model_registry
from hilbert_utils import get_hilbert_order, get_hilbert_inverse
import torch
import torch.nn.functional as F

from merge_observer import compute_merge_token_groups, patch_sam_for_merging


/pfss/mlde/workspaces/mlde_wsp_IAS_SAMMerge/SAM_Quantization/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/pfss/mlde/workspaces/mlde_wsp_IAS_SAMMerge/SAM_Quantization/.venv/lib/python3.10/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/pfss/mlde/workspaces/mlde_wsp_IAS_SAMMerge/SAM_Quantization/.venv/lib/python3.10/site-packages/timm/models/registry.py:4: FutureWarning: Importing from timm.models.registry is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)
/pfss/mlde/workspaces/mlde_wsp_IAS_SAMMerge

In [2]:
model_type = "vit_l"
checkpoint_path = "/pfss/mlde/workspaces/mlde_wsp_IAS_SAMMerge/SAM_Quantization/ckts/sam_hq_vit_l.pth"
image_dir = Path("/pfss/mlde/workspaces/mlde_wsp_IAS_SAMMerge/SAM_Quantization/input_imgs")
number_data = 1
block_idx = 0

merge_sx = 2
merge_sy = 2
gradient_method = "sobel"  # or "central_difference"
device = "cuda" if torch.cuda.is_available() else "cpu"

sam = sam_model_registry[model_type](checkpoint=checkpoint_path).to(device)
observer = patch_sam_for_merging(
    sam,
    gradient_method=gradient_method,
    store_on_cpu=True,
    detach_tensors=True,
)
predictor = SamPredictor(sam)

image_paths = sorted(
    [p for p in image_dir.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"}]
)
print(f"Found {len(image_paths)} images")
print(f"Patched block: image_encoder.blocks.{block_idx}")


<All keys matched successfully>
Found 10 images
Patched block: image_encoder.blocks.0


In [3]:
def load_rgb_image(path: Path):
    image_bgr = cv2.imread(str(path))
    if image_bgr is None:
        raise FileNotFoundError(f"Could not read image: {path}")
    return cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)


def reduce_windows(record):
    edge_flat = record["edge_intensity_flat"].float().mean(dim=0).numpy()
    similarity = record["cosine_similarity"].float().mean(dim=0).numpy()
    h, w = record["spatial_shape"]
    
    order = np.argsort(edge_flat)[::-1]
    similarity_sorted = similarity[np.ix_(order, order)]
    row_similarity = similarity.mean(axis=1)
    
    return {
        "edge_flat": edge_flat,
        "similarity": similarity,
        "similarity_sorted": similarity_sorted,
        "row_similarity": row_similarity,
        "order": order    }


def run_merge_observer(path: Path, block_idx: int):
    observer.clear()
    image_rgb = load_rgb_image(path)
    predictor.set_image(image_rgb)
    record = observer.latest_by_index(block_idx)
    if record is None:
        raise RuntimeError(f"No observer record captured for block {block_idx}")
    return image_rgb, record
def compare_merge_maps(merge_map_attn_np, merge_map_mlp_np, index_value=1):
    """
    Compare two merge maps for a specific class (unm, src, or dst).
    Visualizes class masks and their overlap without RGB image.
    
    Args:
        merge_map_attn_np: First merge map (H, W) with values 0, 1, 2
        merge_map_mlp_np: Second merge map (H, W) with values 0, 1, 2
        index_value: Class to compare (0=unm, 1=src, 2=dst)
    """
    
    class_names = {0: "unm", 1: "src", 2: "dst"}
    class_name = class_names.get(index_value, "unknown")
    
    # Extract masks for the specified class
    mask_attn = (merge_map_attn_np == index_value).astype(np.uint8)
    mask_mlp = (merge_map_mlp_np == index_value).astype(np.uint8)
    
    # Compute metrics
    total_pixels = mask_attn.size
    
    # 1. Hamming distance - count differing pixels (fix: avoid uint8 wraparound)
    hamming = np.sum(mask_attn != mask_mlp)  # Count mismatches directly
    hamming_norm = hamming / total_pixels * 100
    
    # 2. Intersection and Union
    intersection = np.sum(mask_attn & mask_mlp)
    union = np.sum(mask_attn | mask_mlp)
    iou = intersection / union if union > 0 else 0.0
    
    # 3. Dice coefficient
    dice = (2 * intersection) / (np.sum(mask_attn) + np.sum(mask_mlp)) if (np.sum(mask_attn) + np.sum(mask_mlp)) > 0 else 0.0
    
    # 4. Pixel counts
    count_attn = np.sum(mask_attn)
    count_mlp = np.sum(mask_mlp)
    
    # Print metrics
    print("=" * 60)
    print(f"MERGE MAP COMPARISON - CLASS: {class_name.upper()} (index={index_value})")
    print("=" * 60)
    print(f"Total pixels: {total_pixels}")
    print(f"\nAttn {class_name} pixels: {count_attn} ({count_attn/total_pixels*100:.2f}%)")
    print(f"MLP {class_name} pixels:  {count_mlp} ({count_mlp/total_pixels*100:.2f}%)")
    print(f"\n--- Distance Metrics ---")
    print(f"Hamming distance: {hamming} pixels ({hamming_norm:.2f}%)")
    print(f"IoU (Intersection over Union): {iou:.4f}")
    print(f"Dice coefficient: {dice:.4f}")
    print(f"Intersection: {intersection} pixels")
    print(f"Union: {union} pixels")
    print("=" * 60)
    
    # Visualize masks
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    # Attn mask
    axes[0].imshow(mask_attn, cmap="gray")
    axes[0].set_title(f"Attn {class_name} mask\n({count_attn} pixels)")
    axes[0].axis("off")
    
    # MLP mask
    axes[1].imshow(mask_mlp, cmap="gray")
    axes[1].set_title(f"MLP {class_name} mask\n({count_mlp} pixels)")
    axes[1].axis("off")
    
    # Overlap visualization
    overlap = np.zeros((*mask_attn.shape, 3), dtype=np.uint8)
    only_attn = (mask_attn == 1) & (mask_mlp == 0)
    only_mlp = (mask_mlp == 1) & (mask_attn == 0)
    both = (mask_attn == 1) & (mask_mlp == 1)
    
    overlap[only_attn] = [255, 0, 0]    # Red
    overlap[only_mlp] = [0, 255, 0]     # Green
    overlap[both] = [255, 255, 0]       # Yellow
    
    axes[2].imshow(overlap)
    axes[2].set_title(f"Overlap ({class_name})\n(Red=Attn only, Green=MLP only, Yellow=Both)")
    axes[2].axis("off")
    
    plt.tight_layout()
    plt.show()
    
    return {
        "mask_attn": mask_attn,
        "mask_mlp": mask_mlp,
        "hamming": hamming,
        "hamming_norm": hamming_norm,
        "iou": iou,
        "dice": dice,
        "count_attn": count_attn,
        "count_mlp": count_mlp,
        "class_name": class_name,
    }


In [4]:
merge_r = 200
path = image_paths[3]
observer.clear()
image_rgb = load_rgb_image(path)
predictor.set_image(image_rgb)

block_idx = 0
record = observer.latest_by_index(block_idx)
x_attn = record["input_attn_2"]
print(x_attn.shape)



torch.Size([25, 14, 14, 1024])


In [8]:
B, H , W, C = x_attn.shape
x_attn_reshaped = x_attn.view(B, H * W, C)

perm = get_hilbert_order(H, W)
inverse_perm = get_hilbert_inverse(H, W)


In [ ]:
input = x_attn_reshaped[0, perm, :] # T , hd


In [ ]:
merge_r = 200
path = image_paths[3]
observer.clear()
image_rgb = load_rgb_image(path)
predictor.set_image(image_rgb)

index_block_list = [0, 4 ,7 ,11,13, 17]
token_cmap = ListedColormap(["#4c78a8", "#f58518", "#54a24b"])

plt.figure(figsize=(8, 8))
plt.imshow(image_rgb)
plt.title(path.name)
plt.axis("off")
plt.show()

for block_idx in index_block_list:
    record = observer.latest_by_index(block_idx)
    if record is None:
        print(f"No record found for block {block_idx}")
        continue

    edge_map_attn, edge_map_mlp, merge_map_attn, merge_map_mlp = compute_merge_token_groups(
        record["input_attn"],
        record["post_shortcut_x"],
        merge_sx,
        merge_sy,
        gradient_method,
        merge_r,
    )

    edge_map_attn_np = edge_map_attn[0].float().cpu().numpy()
    edge_map_mlp_np = edge_map_mlp[0].float().cpu().numpy()
    merge_map_attn_np = merge_map_attn[0].cpu().numpy()
    merge_map_mlp_np = merge_map_mlp[0].cpu().numpy()
    # compare_merge_maps(merge_map_attn_np, merge_map_mlp_np, index_value=0)  # Compare "src" class
    fig, axes = plt.subplots(2, 2, figsize=(12, 12))
    fig.suptitle(f"Block {block_idx}", fontsize=16)

    im0 = axes[0, 0].imshow(edge_map_attn_np, cmap="magma")
    axes[0, 0].set_title("Edge Map Attn")
    axes[0, 0].axis("off")
    fig.colorbar(im0, ax=axes[0, 0], fraction=0.046)

    im1 = axes[0, 1].imshow(edge_map_mlp_np, cmap="magma")
    axes[0, 1].set_title("Edge Map MLP")
    axes[0, 1].axis("off")
    fig.colorbar(im1, ax=axes[0, 1], fraction=0.046)

    im2 = axes[1, 0].imshow(merge_map_attn_np, cmap=token_cmap, vmin=0, vmax=2)
    axes[1, 0].set_title("Merge Map Attn")
    axes[1, 0].axis("off")
    cbar2 = fig.colorbar(im2, ax=axes[1, 0], ticks=[0, 1, 2], fraction=0.046)
    cbar2.ax.set_yticklabels(["unm", "src", "dst"])

    im3 = axes[1, 1].imshow(merge_map_mlp_np, cmap=token_cmap, vmin=0, vmax=2)
    axes[1, 1].set_title("Merge Map MLP")
    axes[1, 1].axis("off")
    cbar3 = fig.colorbar(im3, ax=axes[1, 1], ticks=[0, 1, 2], fraction=0.046)
    cbar3.ax.set_yticklabels(["unm", "src", "dst"])

    plt.tight_layout()
    plt.show()